# DynamoDB works-graph Table Restore

Rollback plan for restoring the `catalogue-2025-10-02_works-graph` DynamoDB table from a PITR-restored copy.

**Steps:**
1. Clear all items from the corrupted `catalogue-2025-10-02_works-graph` table
2. Scan `restored-catalogue-2025-10-02_works-graph` and batch-write every item into the original table
3. Verify item counts match

**Prerequisites:**
- The PITR restore into `restored-catalogue-2025-10-02_works-graph` must already be complete
- Active AWS SSO session: `aws sso login`

In [ ]:
import os
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

os.environ.setdefault("AWS_PROFILE", "platform-developer")

AWS_PROFILE = os.environ["AWS_PROFILE"]
REGION = "eu-west-1"

CORRUPTED_TABLE = "catalogue-2025-10-02_works-graph"
RESTORED_TABLE = "restored-catalogue-2025-10-02_works-graph"

# Parallel scan segments — increase for larger tables
TOTAL_SEGMENTS = 16

# Max items per DynamoDB BatchWriteItem / BatchDeleteItem call
BATCH_SIZE = 25

# Retry config
MAX_RETRIES = 8
BASE_BACKOFF = 0.1  # seconds

boto_config = Config(
    retries={"max_attempts": 10, "mode": "adaptive"},
    max_pool_connections=TOTAL_SEGMENTS + 4,
)
session = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)
dynamodb = session.resource("dynamodb", config=boto_config)
dynamodb_client = session.client("dynamodb", config=boto_config)

corrupted_table = dynamodb.Table(CORRUPTED_TABLE)
restored_table = dynamodb.Table(RESTORED_TABLE)

print(f"Corrupted table: {CORRUPTED_TABLE}")
print(f"Restored table:  {RESTORED_TABLE}")
print(f"Parallel segments: {TOTAL_SEGMENTS}")

## Helpers

In [ ]:
def exponential_backoff(attempt: int) -> None:
    """Sleep with jittered exponential backoff."""
    delay = BASE_BACKOFF * (2 ** attempt) + random.uniform(0, BASE_BACKOFF)
    time.sleep(delay)


def batch_write_with_retries(table_name: str, items: list[dict]) -> int:
    """Write a batch of items with retry on UnprocessedItems. Returns number written."""
    request_items = {table_name: [{"PutRequest": {"Item": item}} for item in items]}
    written = 0

    for attempt in range(MAX_RETRIES):
        response = dynamodb_client.batch_write_item(
            RequestItems=request_items
        )
        unprocessed = response.get("UnprocessedItems", {})
        processed_count = len(items) - len(unprocessed.get(table_name, []))
        written += processed_count

        if not unprocessed or not unprocessed.get(table_name):
            return written

        # Prepare retry with only the unprocessed items
        request_items = unprocessed
        items = [
            req["PutRequest"]["Item"] for req in unprocessed[table_name]
        ]
        print(f"  Retrying {len(items)} unprocessed items (attempt {attempt + 1})")
        exponential_backoff(attempt)

    raise RuntimeError(
        f"Still have {len(items)} unprocessed items after {MAX_RETRIES} retries"
    )


def batch_delete_with_retries(table_name: str, keys: list[dict]) -> int:
    """Delete a batch of keys with retry on UnprocessedItems. Returns number deleted."""
    request_items = {table_name: [{"DeleteRequest": {"Key": key}} for key in keys]}
    deleted = 0

    for attempt in range(MAX_RETRIES):
        response = dynamodb_client.batch_write_item(
            RequestItems=request_items
        )
        unprocessed = response.get("UnprocessedItems", {})
        processed_count = len(keys) - len(unprocessed.get(table_name, []))
        deleted += processed_count

        if not unprocessed or not unprocessed.get(table_name):
            return deleted

        request_items = unprocessed
        keys = [
            req["DeleteRequest"]["Key"] for req in unprocessed[table_name]
        ]
        print(f"  Retrying {len(keys)} unprocessed deletes (attempt {attempt + 1})")
        exponential_backoff(attempt)

    raise RuntimeError(
        f"Still have {len(keys)} unprocessed deletes after {MAX_RETRIES} retries"
    )


def get_item_count(table_name: str) -> int:
    """Get the approximate item count from table metadata."""
    resp = dynamodb_client.describe_table(TableName=table_name)
    return resp["Table"]["ItemCount"]


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


print("Helpers loaded.")

## Pre-flight checks

Verify both tables exist and print their current item counts.

In [ ]:
corrupted_count = get_item_count(CORRUPTED_TABLE)
restored_count = get_item_count(RESTORED_TABLE)

print(f"Corrupted table ({CORRUPTED_TABLE}): ~{corrupted_count:,} items")
print(f"Restored table  ({RESTORED_TABLE}): ~{restored_count:,} items")
print()
print("⚠️  The next cell will DELETE ALL items from the corrupted table.")
print("    Make sure you are certain before proceeding.")

## Step 1: Clear the corrupted table

Parallel scan + batch delete. The table key is `id` (String).

In [ ]:
def delete_segment(segment: int) -> int:
    """Scan and delete all items in one parallel-scan segment."""
    deleted = 0
    scan_kwargs = {
        "TableName": CORRUPTED_TABLE,
        "ProjectionExpression": "id",
        "TotalSegments": TOTAL_SEGMENTS,
        "Segment": segment,
    }

    while True:
        response = dynamodb_client.scan(**scan_kwargs)
        items = response.get("Items", [])

        if items:
            keys = [{"id": item["id"]} for item in items]
            for batch in chunks(keys, BATCH_SIZE):
                deleted += batch_delete_with_retries(CORRUPTED_TABLE, batch)

        if "LastEvaluatedKey" not in response:
            break
        scan_kwargs["ExclusiveStartKey"] = response["LastEvaluatedKey"]

    return deleted


print(f"Deleting all items from {CORRUPTED_TABLE} using {TOTAL_SEGMENTS} parallel segments...")
start = time.time()
total_deleted = 0

with ThreadPoolExecutor(max_workers=TOTAL_SEGMENTS) as executor:
    futures = {
        executor.submit(delete_segment, seg): seg
        for seg in range(TOTAL_SEGMENTS)
    }
    for future in as_completed(futures):
        seg = futures[future]
        count = future.result()
        total_deleted += count
        print(f"  Segment {seg:2d}: deleted {count:,} items")

elapsed = time.time() - start
print(f"\nDone. Deleted {total_deleted:,} items in {elapsed:.1f}s")

## Step 2: Copy items from restored table to original table

Parallel scan of the restored table + batch write into the original table.

In [ ]:
def copy_segment(segment: int) -> int:
    """Scan one segment of the restored table and write to the corrupted table."""
    copied = 0
    # Use the high-level Table resource for scan so items come back as
    # native Python types (the low-level client returns DynamoDB typed dicts
    # which would need to be converted for batch_write_item).
    scan_kwargs = {
        "TotalSegments": TOTAL_SEGMENTS,
        "Segment": segment,
    }

    while True:
        response = restored_table.scan(**scan_kwargs)
        items = response.get("Items", [])

        if items:
            for batch in chunks(items, BATCH_SIZE):
                with corrupted_table.batch_writer(
                    overwrite_by_pkeys=["id"]
                ) as writer:
                    for item in batch:
                        writer.put_item(Item=item)
                copied += len(batch)

        if "LastEvaluatedKey" not in response:
            break
        scan_kwargs["ExclusiveStartKey"] = response["LastEvaluatedKey"]

    return copied


print(f"Copying items from {RESTORED_TABLE} → {CORRUPTED_TABLE}...")
print(f"Using {TOTAL_SEGMENTS} parallel segments.")
start = time.time()
total_copied = 0

with ThreadPoolExecutor(max_workers=TOTAL_SEGMENTS) as executor:
    futures = {
        executor.submit(copy_segment, seg): seg
        for seg in range(TOTAL_SEGMENTS)
    }
    for future in as_completed(futures):
        seg = futures[future]
        count = future.result()
        total_copied += count
        print(f"  Segment {seg:2d}: copied {count:,} items")

elapsed = time.time() - start
print(f"\nDone. Copied {total_copied:,} items in {elapsed:.1f}s")

## Step 3: Verification

Compare item counts. Note: `ItemCount` from `describe_table` is updated every ~6 hours,
so we do a full count via scan for accuracy.

In [ ]:
def count_items_via_scan(table_name: str) -> int:
    """Count items using a parallel scan (accurate, but slow for large tables)."""
    def count_segment(segment: int) -> int:
        count = 0
        scan_kwargs = {
            "TableName": table_name,
            "Select": "COUNT",
            "TotalSegments": TOTAL_SEGMENTS,
            "Segment": segment,
        }
        while True:
            resp = dynamodb_client.scan(**scan_kwargs)
            count += resp["Count"]
            if "LastEvaluatedKey" not in resp:
                break
            scan_kwargs["ExclusiveStartKey"] = resp["LastEvaluatedKey"]
        return count

    with ThreadPoolExecutor(max_workers=TOTAL_SEGMENTS) as executor:
        counts = list(executor.map(count_segment, range(TOTAL_SEGMENTS)))
    return sum(counts)


print("Counting items (full parallel scan)...")
start = time.time()
final_corrupted = count_items_via_scan(CORRUPTED_TABLE)
final_restored = count_items_via_scan(RESTORED_TABLE)
elapsed = time.time() - start

print(f"  {CORRUPTED_TABLE}: {final_corrupted:,} items")
print(f"  {RESTORED_TABLE}:  {final_restored:,} items")
print(f"  (counted in {elapsed:.1f}s)")
print()

if final_corrupted == final_restored:
    print("✅ Item counts match. Restore looks successful.")
else:
    diff = final_restored - final_corrupted
    print(f"❌ MISMATCH: restored has {diff:,} more items than the target.")
    print("   Investigate before proceeding.")